In [19]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [20]:
from langchain.agents import create_react_agent, AgentExecutor

In [22]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")
GROQ_APU_KEY = os.getenv("GROQ_API_KEY")

In [23]:
search_tool = TavilySearchResults(max_results=2)

In [24]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [25]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Environment & Sustainability\n\nAugust 25, 2026\n\n[](\n\n#### Enterprise\n\n### AI weather forecasting enters the energy market as Google targets grid operators with WeatherNext 3\n\nArtificial Intelligence\n\nSeptember 8, 2026\n\n### OpenAI president urges enterprises to hasten AI security defences\n\nCybersecurity AI\n\nAugust 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n#### Industries\n\n### Google launches Gemini 3.8 Flash TTS voice models\n\nWorld of Work\n\nSeptember 24, 2026\n\n### NVIDIA Jetson Orin Nano 2 brings physical AI to drones and robots\n\nPhysical AI\n\nAugust 26, 2026\n\n### HoneyBook bets on agentic AI to streamline small business operations with its new Claude connector\n\nWorld of Work\n\nAugust 6, 2026\n\n#### Deep Learning [...] Finance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely av

In [38]:
# llm = ChatOpenAI(
#     model="nvidia/nemotron-3.5-lightning-30b-a3b",
#     base_url="https://integrate.api.nvidia.com/v1",
#     api_key=OPENAI_API_KEY,
#     temperature=1,
#     max_tokens=4096,
#     streaming=True,
#     model_kwargs={
#         "top_p": 0.95
#     }
# )

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
    max_tokens=1024,
    streaming=False,
)

In [39]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI break up with the robot?\n\nBecause it felt there was no **spark** between them. ⚡️🤖', response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 18, 'total_tokens': 49, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'queue_time': 0.048043444, 'prompt_time': 0.001014446, 'completion_time': 0.059128092, 'total_time': 0.060142538}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'finish_reason': 'stop', 'logprobs': None}, id='run-c28fd4c0-3ac0-4cca-99dd-afc2730bc7df-0')

In [40]:
prompt = hub.pull("hwchase17/react")

d:\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [41]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [42]:
tools = [search_tool, get_weather_data]

In [43]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [44]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True

)

In [47]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of India and number of trees in the city"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
Question: Find the capital of India and number of trees in the city and then find its current weather.
Thought: First, I need to identify the capital of India. I know it is New Delhi, but I will verify this and also search for the number of trees in New Delhi to ensure accuracy.
Action: tavily_search_results_json
Action Input: capital of India and number of trees in New Delhi[{'url': 'https://supersanyam.medium.com/guesstimate-estimate-the-number-of-trees-in-delhi-32e63451163', 'content': 'Number of trees in Delhi. Number of trees in Delhi = ~55000 * 1500 = 8.25 Crore Trees'}, {'url': 'https://www.facebook.com/India.Architecture/posts/new-delhi-india-did-you-know-new-delhi-is-one-of-the-greenest-capitals-in-the-wo/1286920676806911', 'content': 'New Delhi is one of the greenest capitals in the world — nearly 20% of its area is covered with trees and parks! with many small forests, parks'}]Thought: I have confirmed that the capital of India is New 

In [48]:
print(response["output"])

The capital of India is New Delhi. The city has approximately 8.25 Crore (82.5 million) trees. The current weather in New Delhi is Overcast with a temperature of 24°C (feels like 25°C), humidity at 70%, and wind speed of 13 km/h from the ENE.
